In [1]:
import win32com.client
import pandas as pd
import numpy as np
import xlwings as xw
import re
import time
from datetime import datetime, timedelta
from datetime import date
import datetime
import os
import shutil
from openpyxl import load_workbook
import getpass
from time import strftime
import xlrd

In [2]:
import polars as pl

In [3]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages

In [4]:
InIcIo = datetime.datetime.now()

In [5]:
fecha_hoy = datetime.datetime.today().strftime('%d.%m.%y')

## Creando los P*Q

In [6]:
Qest = "Z:\\Benchmarking\\QEST_BD.xlsm"
# Monitor = r'\\sppeapp00023\\Inversiones\\Mesa de Inversiones\\Bottom-Up\\5 Estrategia del Portafolio\\Monitores\\Monitor Múltiplos RVL\\Monitor Z Valuation RVL.xlsm"
Monitor = r"Z:\Mesa de Inversiones\Bottom-Up\5 Estrategia del Portafolio\Monitores\ZODA\ZODA.xlsm"

In [7]:
##Activos que se van a agregar
#[IFSUS, IFS PE]
IFS = ['300013200002','300013200001']
#[Pacas PE, Pacas ADR, Pacas Inv]
Pacasmayo = ['302051100001','3220512Q1094','312051100001']
#[BVN, Buenaventura]
BVN = ['3220082A1040','302008100002']
#[Volcan B, Volcan A]
Volcan = ['302043100002','302043100001']
#[Alicorp,Alicorp Inv]
Alicorp = ['302011100001','312011100001']
#[Aenza, Aenza US]
Aenza =['302029100001','3220292P2083']

In [8]:
def suma_de_Cols_No_Convertibles (df, columnas):
    a=columnas[0]
    if columnas[0] not in df.columns:
        df[a]=0
    else:
        df[a]=df[a].fillna(0)
    for col in columnas:
        if col != a:
            if col in df.columns:
                df[col]=df[col].fillna(0)
            else:
                df[col]=0
                #print(col)
                
            df[a] = df[a]+df[col]
            
def elimina_sumados (df,columnas):
    a=columnas[0]
    for col in columnas:
        if col != a:
            df.drop(columns=col, inplace=True)
            #print(col)b
            

In [9]:
def DQ_IN(df1,DQ_IN_BD):

    ListaIN = ["Fondo","Dato","Fecha","SBS", '302012200001' , '300013200002' , '300013200001' , '300011100001' , '300009100001' , '302002100001' , '302051100001' , '312051100001' , '3220512Q1094' , '302029100001' , '3220082A1040' , '302008100002' , '302072200001' , '302845100002' , '395833228291' , '312039100001' , '302043100001' , '302043100002' , '302032100002' , '302030100001' , '302061100001' , '302011100001' , '312011100001' , '302107100001' , '3021012171A1' , '345874294721' , '312025100001' , '312034100001' , '312026100001' , '3020836W5029' , '3220292P2083']

    #Se eliminan las primeras filas del documento que no serán usadas
    df1 = df1.drop(df1.index[list(range(0, 10))])
    df1 = df1.transpose()
    df1 =df1.rename(columns={10:"Isins"})
    filtrada=df1[df1["Isins"].isin(ListaIN)]
    filtrada = filtrada.transpose()
    filtrada1 = filtrada
    filtrada1.columns=filtrada.iloc[0].to_list()
    filtrada = filtrada1[1:]

    DQ_IN_FECHAS = DQ_IN_BD["Fecha"][::-1].copy()
    DQ_IN_UltimaFecha = DQ_IN_FECHAS.iloc[0]


    print("DQ IN: " + str(DQ_IN_UltimaFecha))

    Qest_DQ_IN_Fechas = filtrada["Fecha"].values.tolist()

    ubifechas = Qest_DQ_IN_Fechas.index(DQ_IN_UltimaFecha)+9
    Nuevos_DQ_IN=filtrada[ubifechas::]
    Nuevos_DQ_IN=Nuevos_DQ_IN.reset_index()
    Nuevos_DQ_IN =Nuevos_DQ_IN.drop("index",axis=1)
    Nuevos_DQ_IN = Nuevos_DQ_IN.loc[Nuevos_DQ_IN['Dato'].str.contains(r'Q|PxQ')]
    
    ListaUsados = ["Fondo","Dato","Fecha","SBS", '302012200001' , '300013200002' , '300013200001' , '300011100001' , '300009100001' , '302002100001' , '302051100001' , '312051100001' , '3220512Q1094' , '302029100001' , '3220082A1040' , '302008100002' , '302072200001' , '302845100002' , '395833228291' , '312039100001' , '302043100001' , '302043100002' , '302032100002' , '302030100001' , '302061100001' , '302011100001' , '312011100001' , '302107100001' , '3021012171A1' , '345874294721' , '312025100001' , '312034100001' , '312026100001' , '3020836W5029' , '3220292P2083']

    LISTO = pd.concat([DQ_IN_BD,Nuevos_DQ_IN])

    return LISTO

In [10]:
import fastexcel

In [11]:
DQ_IN_BD=pl.read_excel(Monitor, sheet_name="DQ_IN").to_pandas()
df1=pl.read_excel(Qest, sheet_name="DQ_IN", has_header=False).to_pandas()
Base_Integra = DQ_IN(df1,DQ_IN_BD)

DQ IN: 2026-02-10 00:00:00


In [12]:
Base_Integra=Base_Integra.reset_index(drop=True)
Base_Integra.loc[3:,"Fecha"]=pd.to_datetime(Base_Integra.loc[3:,"Fecha"], dayfirst=True)
Emisores_Usados=['Fecha','302012200001' , '300013200002' , '300013200001' , '300011100001' , '300009100001' , '302002100001' , '302051100001' , '312051100001' , '3220512Q1094' , '302029100001' , '3220082A1040' , '302008100002' , '302072200001' , '302845100002' , '395833228291' , '312039100001' , '302043100001' , '302043100002' , '302032100002' , '302030100001' , '302061100001' , '302011100001' , '312011100001' , '302107100001' , '3021012171A1' , '345874294721' , '312025100001' , '312034100001' , '312026100001' , '3020836W5029' , '3220292P2083']
AFloat=['302012200001' , '300013200002' , '300013200001' , '300011100001' , '300009100001' , '302002100001' , '302051100001' , '312051100001' , '3220512Q1094' , '302029100001' , '3220082A1040' , '302008100002' , '302072200001' , '302845100002' , '395833228291' , '312039100001' , '302043100001' , '302043100002' , '302032100002' , '302030100001' , '302061100001' , '302011100001' , '312011100001' , '302107100001' , '3021012171A1' , '345874294721' , '312025100001' , '312034100001' , '312026100001' , '3020836W5029' , '3220292P2083']

for i in AFloat:
    Base_Integra.loc[3:,i]=Base_Integra.loc[3:,i].astype(np.float64)

C:\Users\P048513\AppData\Local\Temp\ipykernel_28008\484212526.py:2: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  Base_Integra.loc[3:,"Fecha"]=pd.to_datetime(Base_Integra.loc[3:,"Fecha"], dayfirst=True)


# Creando la BD

In [13]:
Data = r"Z:\Mesa de Inversiones\Bottom-Up\5 Estrategia del Portafolio\Monitores\ZODA\ZODA.xlsm"

Financials_Datos= pl.read_excel(Data, sheet_name="Financials", engine="calamine",has_header=False,read_options={ "skip_rows":0,"n_rows":7600}).to_pandas()
Financials_Datos=Financials_Datos.copy()

Min_Utl_Datos= pl.read_excel(Data, sheet_name="Min y Uti", engine="calamine",has_header=False,read_options={ "skip_rows":0,"n_rows":7600}).to_pandas()
Min_Utl_Datos=Min_Utl_Datos.copy()

Cons_Const_Datos= pl.read_excel(Data, sheet_name="Cons y Const", engine="calamine",has_header=False,read_options={ "skip_rows":0,"n_rows":7600}).to_pandas()
Cons_Const_Datos=Cons_Const_Datos.copy()

Indices_Datos= pl.read_excel(Data, sheet_name="Indices", engine="calamine",has_header=False,read_options={ "skip_rows":0,"n_rows":7600}).to_pandas()
Indices_Datos=Indices_Datos.copy()

Could not determine dtype for column 81, falling back to string
Could not determine dtype for column 82, falling back to string
Could not determine dtype for column 249, falling back to string
Could not determine dtype for column 250, falling back to string
Could not determine dtype for column 357, falling back to string
Could not determine dtype for column 358, falling back to string
Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 21, falling back to string
Could not determine dtype for column 32, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 54, falling back to string


## Pesos a darles a los distintos años de Z's, y a peso a los multiplos.

In [14]:
Anios_Z=[1,3,5,10]
Pesos_Z=[0.33,0.33,0.33,0]

#                 Trail P/E, Fwd P/E, Trail P/S, Fwd P/S, Trail P/B, Fwd P/B, Trail EV/EBITDA, Fwd EV/EBITDA
Pesos_mult_Finan = [0.33, 0, 0.33, 0, 0.33, 0, 0, 0]
Pesos_mult_Ex_Finan = [0.25, 0, 0.25, 0, 0.25, 0, 0.25, 0]


## Listas de Acciones y sus Baskets

In [15]:
Baskets= pl.read_excel(Data, sheet_name="Baskets", engine="calamine",has_header=False,read_options={ "skip_rows":0,"n_rows":100}).to_pandas()
Baskets_O=Baskets.copy()
Baskets_O=Baskets_O.dropna(axis=1, how= "all").copy()
Filas_Baskets = ["Con, Ind y HC","Mineria y Utilities","Financieras","Indices"]
Baskets_Grupos= {}

In [16]:
#### Con, Ind y HC
Baskets_Con_Ind_HC = Baskets_O[Baskets_O.iloc[:, 0] == "Con, Ind y HC"]
Baskets_Con_Ind_HC = Baskets_Con_Ind_HC.drop(Baskets_Con_Ind_HC.columns[0], axis=1)
Baskets_Con_Ind_HC=Baskets_Con_Ind_HC.dropna(axis=1, how= "all").copy()

#### Todo
Baskets_Con_Ind_HC_Todo_List = Baskets_Con_Ind_HC.values.flatten()
Baskets_Con_Ind_HC_Todo_List = [item for item in Baskets_Con_Ind_HC_Todo_List if item is not None and item != 'None']
Baskets_Con_Ind_HC_Todo_List_O = list(dict.fromkeys(Baskets_Con_Ind_HC_Todo_List))
#### Main
Baskets_Con_Ind_HC_Emp = Baskets_Con_Ind_HC.iloc[0:, 0]
Baskets_Con_Ind_HC_Emp_List = Baskets_Con_Ind_HC_Emp.values.flatten()
Baskets_Con_Ind_HC_Emp_List_O = list(dict.fromkeys(Baskets_Con_Ind_HC_Emp_List))
#### Basket
Baskets_Con_Ind_HC_Dic = {}
for empresas in Baskets_Con_Ind_HC_Emp_List_O:
    index= Baskets_Con_Ind_HC_Emp.tolist().index(empresas)
    Comparables_Derecha = Baskets_Con_Ind_HC.iloc[index:,1:]
    Comparables_Derecha_List = list(dict.fromkeys(Comparables_Derecha.values.flatten()))
    Comparables_Derecha_List = [v for v in Comparables_Derecha_List if v is not None and v != 'None']
    Baskets_Con_Ind_HC_Dic[empresas] = Comparables_Derecha_List


In [17]:
#### Mineria y Utilities
Baskets_Min_Uti = Baskets_O[Baskets_O.iloc[:, 0] == "Mineria y Utilities"]
Baskets_Min_Uti = Baskets_Min_Uti.drop(Baskets_Min_Uti.columns[0], axis=1)
Baskets_Min_Uti=Baskets_Min_Uti.dropna(axis=1, how= "all").copy()

#### Todo
Baskets_Min_Uti_Todo_List = Baskets_Min_Uti.values.flatten()
Baskets_Min_Uti_Todo_List = [item for item in Baskets_Min_Uti_Todo_List if item is not None and item != 'None']
Baskets_Min_Uti_Todo_List_O = list(dict.fromkeys(Baskets_Min_Uti_Todo_List))
#### Main
Baskets_Min_Uti_Emp = Baskets_Min_Uti.iloc[0:, 0]
Baskets_Min_Uti_Emp_List = Baskets_Min_Uti_Emp.values.flatten()
Baskets_Min_Uti_Emp_List_O = list(dict.fromkeys(Baskets_Min_Uti_Emp_List))
#### Basket
Baskets_Min_Uti_Emp_Dic = {}
for empresas in Baskets_Min_Uti_Emp_List_O:
    index= Baskets_Min_Uti_Emp.tolist().index(empresas)
    Comparables_Derecha = Baskets_Min_Uti.iloc[index:,1:]
    Comparables_Derecha_List = list(dict.fromkeys(Comparables_Derecha.values.flatten()))
    Comparables_Derecha_List = [v for v in Comparables_Derecha_List if v is not None and v != 'None']
    Baskets_Min_Uti_Emp_Dic[empresas] = Comparables_Derecha_List

In [18]:
#### Financieras
Baskets_Financieras = Baskets_O[Baskets_O.iloc[:, 0] == "Financieras"]
Baskets_Financieras = Baskets_Financieras.drop(Baskets_Financieras.columns[0], axis=1)
Baskets_Financieras=Baskets_Financieras.dropna(axis=1, how= "all").copy()

#### Todo
Baskets_Financieras_Todo_List = Baskets_Financieras.values.flatten()
Baskets_Financieras_Todo_List = [item for item in Baskets_Financieras_Todo_List if item is not None and item != 'None']
Baskets_Financieras_Todo_List_O = list(dict.fromkeys(Baskets_Financieras_Todo_List))
#### Main
Baskets_Financieras_Emp = Baskets_Financieras.iloc[0:, 0]
Baskets_Financieras_Emp_List = Baskets_Financieras_Emp.values.flatten()
Baskets_Financieras_Emp_List_O = list(dict.fromkeys(Baskets_Financieras_Emp_List))
#### Basket
Baskets_Financieras_Emp_Dic = {}
for empresas in Baskets_Financieras_Emp_List_O:
    index= Baskets_Financieras_Emp.tolist().index(empresas)
    Comparables_Derecha = Baskets_Financieras.iloc[index:,1:]
    Comparables_Derecha_List = list(dict.fromkeys(Comparables_Derecha.values.flatten()))
    Comparables_Derecha_List = [v for v in Comparables_Derecha_List if v is not None and v != 'None']
    Baskets_Financieras_Emp_Dic[empresas] = Comparables_Derecha_List

In [19]:
Todas_Cons_Const = Baskets_Con_Ind_HC_Todo_List_O
Empresas_Cons_Const = Baskets_Con_Ind_HC_Emp_List_O
Basket_Cons_Const = Baskets_Con_Ind_HC_Dic

Todas_Min_Utl = Baskets_Min_Uti_Todo_List_O
Empresas_Min_Utl = Baskets_Min_Uti_Emp_List_O
Basket_Min_Utl = Baskets_Min_Uti_Emp_Dic

Todas_Finan = Baskets_Financieras_Todo_List_O
Empresas_Financieras = Baskets_Financieras_Emp_List_O
Basket_Financieras = Baskets_Financieras_Emp_Dic

In [20]:
Todas_Indices=['SPBLPGPT Index','IPSA Index',"IBOV Index","COLCAP Index","MEXBOL Index","MXLA Index"]

In [21]:
# Todas_Finan=["BAP US Equity","IFS PE Equity","BBVAC1 PE Equity",'GFNORTEO MM Equity','BBVA* MM Equity','GFINBURO MM Equity','BSMXB MM Equity','GENTERA* MM Equity','ITUB3 BZ Equity','BBDC3 BZ Equity','BBAS3 BZ Equity','SANB11 BZ Equity','BPAC3 BZ Equity','CHILE CI Equity','BSAN CI Equity','BCI CI Equity','ITAUCL CC Equity','BCOLO CB Equity','BOGOTA CB Equity','PFDAVVND CB Equity']

# Todas_Cons_Const=['ALICORC1 PE Equity','INRETC1 PE Equity','BACKUSI1 PE Equity','CORAREC1 PE Equity','CPACASC1 PE Equity','AENZAC1 PE Equity','FERREYC1 PE Equity','ELCOMEI1 PE Equity','AUNA US Equity','NUTRESA CB Equity','GLORIAI1 PE Equity','GRUMAB MM Equity','MDIA3 BZ Equity','BIMBOA MM Equity','EBRO SM Equity','TBS SJ Equity','CAML3 BZ Equity','HERDEZ* MM Equity','FALAB CI Equity','CENCOSUD CI Equity','WALMEX* MM Equity','SORIANAB MM Equity','CA FP Equity','GMAT3 BZ Equity','EXITO CB Equity','NTCO3 BZ Equity','PNVL3 BZ Equity','CCU CI Equity','CONCHA CI Equity','FEMSAUBD MM Equity','ANDINAB CI Equity','HEIA NA Equity','ABEV3 BZ Equity','ABI BB Equity','AC* MM Equity','6176 JT Equity','STAR MK Equity','423 HK Equity','033130 KS Equity','BST GR Equity','3560 JT Equity','DMS US Equity','HMVL IN Equity','GGBR4 BZ Equity','ZEUS US Equity','TXAR AR Equity','SIDERC1 PE Equity','FESA3 BZ Equity','CINTAC CI Equity','UNACEMC1 PE Equity','CEMEXCPO MM Equity','CEMARGOS CB Equity','EXP US Equity','GCC* MM Equity','CEMENT CC Equity','HOLN SW Equity','MELON CI Equity','LOMA US Equity','SALFACOR CI Equity','BESALCO CI Equity','SK CC Equity','EISA CI Equity','IDEALB1 MM Equity','FCC SM Equity','OHLA SM Equity','RAPT4 BZ Equity','WJX CN Equity','TITN US Equity','FTT CN Equity','BAW SJ Equity','TIH CN Equity','SVW AU Equity','HEES US Equity','RDOR3 BZ Equity','HAPV3 BZ Equity','MATD3 BZ Equity','ONCO3 BZ Equity','FLRY3 BZ Equity','DASA3 BZ Equity','KRSA3 BZ Equity','MEDICAB MF Equity','TRE SM Equity','SCYR SM Equity']

# Todas_Min_Utl=['CVERDEC1 PE Equity','AAL LN Equity','SCCO US Equity','FCX US Equity','ANTO LN Equity','FM CN Equity','CS CN Equity','PUCOBRE CI Equity','1208 HK Equity','HBM CN Equity','MINSURI1 PE Equity','000960 CH Equity','TINS IJ Equity','MLX AU Equity','SMELT MK Equity','NEXA US Equity','BHP LN Equity','BOL SS Equity','TECK US Equity','HZ IN Equity','RIO LN Equity','MFRISCOA MM Equity','GLEN LN Equity','NEXAPEC1 PE Equity','VOLCABC1 PE Equity','BVN US Equity','NEM US Equity','GOLD US Equity','AEM US Equity','GFI US Equity','SSRM US Equity','KGC US Equity','HOC LN Equity','FRES LN Equity','PAAS CN Equity','AG CN Equity','WPM CN Equity','PE&OLES* MM Equity','FVI CN Equity','ORYGENC1 PE Equity','ENELAM CI Equity','ELET3 BZ Equity','EGIE3 BZ Equity','EQTL3 BZ Equity','ENELGXCH CI Equity','CPLE6 BZ Equity','CMIG4 BZ Equity','COLBUN CC Equity','AESANDES CI Equity','ENGIEC1 PE Equity','PLUZENC1 PE Equity','LUSURC1 PE Equity','ENELDXCH CI Equity','ENGI11 BZ Equity','PAM US Equity','CESC IN Equity','PNW US Equity','COCE5 BZ Equity']
# Todas_Indices=['SPBLPGPT Index','IPSA Index',"IBOV Index","COLCAP Index","MEXBOL Index","MXLA Index"]

 
# Empresas_Cons_Const =["ALICORC1 PE Equity","INRETC1 PE Equity","BACKUSI1 PE Equity","CORAREC1 PE Equity","CPACASC1 PE Equity","UNACEMC1 PE Equity","AENZAC1 PE Equity","FERREYC1 PE Equity","ELCOMEI1 PE Equity","AUNA US Equity"]

# Basket_Cons_Const= {"ALICORC1 PE Equity":['NUTRESA CB Equity','GLORIAI1 PE Equity','GRUMAB MM Equity','MDIA3 BZ Equity','BIMBOA MM Equity','EBRO SM Equity','TBS SJ Equity','CAML3 BZ Equity','HERDEZ* MM Equity']

#                  ,"INRETC1 PE Equity":['FALAB CI Equity','CENCOSUD CI Equity','WALMEX* MM Equity','SORIANAB MM Equity','CA FP Equity','GMAT3 BZ Equity','EXITO CB Equity','NTCO3 BZ Equity','PNVL3 BZ Equity']

#                  ,"BACKUSI1 PE Equity":['6176 JT Equity','STAR MK Equity','423 HK Equity','033130 KS Equity','BST GR Equity','3560 JT Equity','DMS US Equity','HMVL IN Equity']

#                  ,"CORAREC1 PE Equity":['GGBR4 BZ Equity','ZEUS US Equity','TXAR AR Equity','SIDERC1 PE Equity','FESA3 BZ Equity','CINTAC CI Equity']

#                  ,"CPACASC1 PE Equity":['UNACEMC1 PE Equity','CEMEXCPO MM Equity','CEMARGOS CB Equity','EXP US Equity','GCC* MM Equity','CEMENT CC Equity','HOLN SW Equity','MELON CI Equity','LOMA US Equity']

#                  ,"UNACEMC1 PE Equity":['CPACASC1 PE Equity','CEMEXCPO MM Equity','CEMARGOS CB Equity','EXP US Equity','GCC* MM Equity','CEMENT CC Equity','HOLN SW Equity','MELON CI Equity','LOMA US Equity']

#                  ,"AENZAC1 PE Equity":['SALFACOR CI Equity','BESALCO CI Equity','SK CC Equity','EISA CI Equity','IDEALB1 MM Equity','TRE SM Equity','FCC SM Equity','OHLA SM Equity','SCYR SM Equity']

#                  ,"FERREYC1 PE Equity":['RAPT4 BZ Equity','WJX CN Equity','TITN US Equity','FTT CN Equity','BAW SJ Equity','TIH CN Equity','SVW AU Equity','HEES US Equity']

#                  ,"ELCOMEI1 PE Equity":['6176 JT Equity','STAR MK Equity','423 HK Equity','033130 KS Equity','BST GR Equity','3560 JT Equity','DMS US Equity','HMVL IN Equity']
#                  ,"AUNA US Equity":['RDOR3 BZ Equity', 'HAPV3 BZ Equity', 'MATD3 BZ Equity', 'ONCO3 BZ Equity', 'DASA3 BZ Equity', 'MEDICAB MF Equity']}
 
# Empresas_Min_Utl=["CVERDEC1 PE Equity","MINSURI1 PE Equity","NEXA US Equity","NEXAPEC1 PE Equity","VOLCABC1 PE Equity","BVN US Equity","HOC LN Equity","ORYGENC1 PE Equity","PLUZENC1 PE Equity","ENGIEC1 PE Equity"]

# Basket_Min_Utl={"CVERDEC1 PE Equity":['AAL LN Equity','SCCO US Equity','FCX US Equity','ANTO LN Equity','FM CN Equity','CS CN Equity','PUCOBRE CI Equity','1208 HK Equity','HBM CN Equity']

#                 ,"MINSURI1 PE Equity":['000960 CH Equity','TINS IJ Equity','MLX AU Equity','SMELT MK Equity','SCCO US Equity','ANTO LN Equity','PUCOBRE CI Equity']
#                 ,"NEXA US Equity":['BHP LN Equity','BOL SS Equity','TECK US Equity','HZ IN Equity','RIO LN Equity','MFRISCOA MM Equity','MFRISCOA MM Equity','GLEN LN Equity']
#                 ,"NEXAPEC1 PE Equity":['BHP LN Equity','BOL SS Equity','TECK US Equity','HZ IN Equity','RIO LN Equity','MFRISCOA MM Equity','MFRISCOA MM Equity','GLEN LN Equity']

#                 ,"VOLCABC1 PE Equity":['BHP LN Equity','BOL SS Equity','TECK US Equity','HZ IN Equity','RIO LN Equity','MFRISCOA MM Equity','MFRISCOA MM Equity','GLEN LN Equity']

#                 ,"BVN US Equity":['NEM US Equity','GOLD US Equity','AEM US Equity','GFI US Equity','SSRM US Equity','KGC US Equity']

#                 ,"HOC LN Equity":['FRES LN Equity','PAAS CN Equity','AG CN Equity','WPM CN Equity','PE&OLES* MM Equity','FVI CN Equity']

#                 ,"ORYGENC1 PE Equity":['ENELAM CI Equity','ELET3 BZ Equity','EGIE3 BZ Equity','EQTL3 BZ Equity','ENELGXCH CI Equity','CPLE6 BZ Equity','CMIG4 BZ Equity','COLBUN CC Equity','AESANDES CI Equity']

#                 ,"PLUZENC1 PE Equity":['LUSURC1 PE Equity','ENELDXCH CI Equity','ENGI11 BZ Equity','PAM US Equity','CESC IN Equity','PNW US Equity','COCE5 BZ Equity']

#                 ,"ENGIEC1 PE Equity":['ENELAM CI Equity','ELET3 BZ Equity','EGIE3 BZ Equity','EQTL3 BZ Equity','ENELGXCH CI Equity','CPLE6 BZ Equity','CMIG4 BZ Equity','COLBUN CC Equity','AESANDES CI Equity']}
 
# Empresas_Financieras=["BAP US Equity","IFS PE Equity","BBVAC1 PE Equity"]

# Basket_Financieras={"BAP US Equity":['GFNORTEO MM Equity','BBVA* MM Equity','GFINBURO MM Equity','BSMXB MM Equity','GENTERA* MM Equity','ITUB3 BZ Equity','BBDC3 BZ Equity','BBAS3 BZ Equity','SANB11 BZ Equity','BPAC3 BZ Equity','CHILE CI Equity','BSAN CI Equity','BCI CI Equity','ITAUCL CC Equity','BCOLO CB Equity','BOGOTA CB Equity','PFDAVVND CB Equity']
#                     ,"IFS PE Equity":['GFNORTEO MM Equity','BBVA* MM Equity','GFINBURO MM Equity','BSMXB MM Equity','GENTERA* MM Equity','ITUB3 BZ Equity','BBDC3 BZ Equity','BBAS3 BZ Equity','SANB11 BZ Equity','BPAC3 BZ Equity','CHILE CI Equity','BSAN CI Equity','BCI CI Equity','ITAUCL CC Equity','BCOLO CB Equity','BOGOTA CB Equity','PFDAVVND CB Equity']
#                     ,"BBVAC1 PE Equity":['GFNORTEO MM Equity','BBVA* MM Equity','GFINBURO MM Equity','BSMXB MM Equity','GENTERA* MM Equity','ITUB3 BZ Equity','BBDC3 BZ Equity','BBAS3 BZ Equity','SANB11 BZ Equity','BPAC3 BZ Equity','CHILE CI Equity','BSAN CI Equity','BCI CI Equity','ITAUCL CC Equity','BCOLO CB Equity','BOGOTA CB Equity','PFDAVVND CB Equity']}
 

Se esta trabajando para que se puedan dar pesos customizados para las empresas que se quiera

## Inicio del Código

In [22]:
def forwards_one_year(Fechas_BD, Base, names):
    Fechas_BD = pd.Series(Base.index.to_list())
    
    Actual_F = pd.DataFrame(Fechas_BD, columns=["Fecha"])
    
    date_to_index = {date: idx for idx, date in enumerate(Fechas_BD)}
    
    Doce_meses = {name: [None] * len(Fechas_BD) for name in names}
    max_fecha = Fechas_BD.max()
    Tasa = 0  
    
    for idx, dia in enumerate(Fechas_BD):
        forward_doce_meses = dia + pd.DateOffset(months=12)
        forward_doce_meses = forward_doce_meses.replace(hour=0, minute=0, second=0)
        
        for name in names:
            if forward_doce_meses in date_to_index:
                data_doce_meses = Base.iloc[date_to_index[forward_doce_meses], names.index(name)]
            else:
                previous_dates = Fechas_BD[Fechas_BD < forward_doce_meses]
                if not previous_dates.empty:
                    closest_date = previous_dates.max()
                    data_doce_meses = Base.iloc[date_to_index[closest_date], names.index(name)]
                    if forward_doce_meses > max_fecha:
                        diff_dias = (forward_doce_meses - max_fecha).days
                        data_doce_meses *= (1 + Tasa) ** (diff_dias / 365)
                else:
                    data_doce_meses = None
            
            Doce_meses[name][idx] = data_doce_meses
    
    Doce_meses_df = pd.DataFrame(Doce_meses)
    Final_fwd = pd.concat([Actual_F,Doce_meses_df], axis=1)
    Final_fwd.set_index("Fecha", inplace=True)
    Final_fwd=pd.concat([Base,Final_fwd],axis=1)
    
    return Final_fwd

In [23]:
Fechas_Tri = Financials_Datos["column_1"][5:]
Fechas_Tri=pd.to_datetime(Fechas_Tri, format="%Y-%m-%d %H:%M:%S")
Fechas_Tri.dropna(inplace=True)

In [24]:
Fechas_Diarias = Financials_Datos.loc[:,Financials_Datos.iloc[2] == "Px_last"].iloc[5:,0]
Fechas_Diarias=pd.to_datetime(Fechas_Diarias, format="%Y-%m-%d %H:%M:%S")
Fechas_Diarias.dropna(inplace=True)

In [25]:
## Limpieza

In [26]:
def LimpiandoCIQ(Base,Fechas,Nombre):
    Data_empresa=Base.loc[:,Base.iloc[4]==Nombre]
    columnas=Data_empresa.loc[2].tolist()
    Data_empresa_=Data_empresa[5:]
    Data_empresa_.columns= columnas
    Data_CIQ=Data_empresa_.iloc[::,:4].dropna(axis=0,how="all")##Aqui me estoy quedando con las 4 columnas de CIQ
    Data_CIQ=Data_CIQ.fillna(0)
    Data_CIQ.insert(0,"Fecha", Fechas)
    Data_CIQ.set_index("Fecha", inplace= True)
    Data_CIQ.replace("#N/A N/A",0, inplace=True)
    Data_CIQ.replace("NM",0, inplace=True)
    Data_CIQ.replace("(Invalid Formula Name)",0, inplace=True)
    Data_CIQ.replace("(Invalid Formula Name)",0, inplace=True)
    Data_CIQ.replace("NA",0, inplace=True)
    Data_CIQ=Data_CIQ.astype(np.float64)
    ##Creando Forwards
    Data_CIQ=forwards_one_year(Fechas_Tri,Data_CIQ,["Fwd_IQ_NI","Fwd_IQ_TOTAL_REV","Fwd_IQ_TOTAL_EQUITY","Fwd_IQ_EBITDA"])
    
    
    return Data_CIQ

In [27]:
def LimpiandoBBG(Base,Fechas,Nombre):
    Data_empresa=Base.loc[:,Base.iloc[4]==Nombre]
    columnas=Data_empresa.loc[2].tolist()
    Data_empresa_=Data_empresa[5:]
    Data_empresa_.columns= columnas
    Data_BBG=Data_empresa_.iloc[::,4:].dropna(axis=0,how="all")
    Data_BBG=Data_BBG.fillna(0)
    Data_BBG.insert(0,"Fecha", Fechas)
    Data_BBG.set_index("Fecha", inplace= True)
    Data_BBG.replace("#N/A N/A",0, inplace=True)
    Data_BBG.replace("NM",0, inplace=True)
    Data_BBG.replace("(Invalid Formula Name)",0, inplace=True)
    Data_BBG.replace("NA",0, inplace=True)
    Data_BBG=Data_BBG.astype(np.float64)
    
    
    return Data_BBG

In [28]:
def LimpiandoIndices(Base,Fechas,Nombre):
    Data_Indice=Base.loc[:,Base.iloc[3]==Nombre]
    columnas=["PX_LAST","Trail_PE","Fwd_PE","Trail_PS","Fwd_PS","Trail_PB","Fwd_PB","Trail_EVEBITDA","Fwd_EVEBITDA"]
    Data_Indice=Data_Indice[5:]
    Data_Indice.columns= columnas
    Data_Indice=Data_Indice.fillna(0)
    Data_Indice.insert(0,"Fecha", Fechas)
    Data_Indice.set_index("Fecha", inplace= True)
    Data_Indice.replace("#N/A N/A",0, inplace=True)
    Data_Indice.replace("NM",0, inplace=True)
    Data_Indice.replace("(Invalid Formula Name)",0, inplace=True)
    Data_Indice.replace("NA",0, inplace=True)
    Data_Indice=Data_Indice.astype(np.float64)
    
    
    
    return Data_Indice

In [29]:
    Data_Indice=Indices_Datos.loc[:,Indices_Datos.iloc[3]=="SPBLPGPT Index"]
    columnas=["PX_LAST","Trail_PE","Fwd_PE","Trail_PS","Fwd_PS","Trail_PB","Fwd_PB","Trail_EVEBITDA","Fwd_EVEBITDA"]
    Data_Indice=Data_Indice[5:]
    Data_Indice.columns= columnas
    Data_Indice=Data_Indice.fillna(0)
    Data_Indice.insert(0,"Fecha", Fechas_Diarias)
    Data_Indice.set_index("Fecha", inplace= True)
    Data_Indice.replace("#N/A N/A",0, inplace=True)
    Data_Indice.replace("NM",0, inplace=True)
    Data_Indice.replace("(Invalid Formula Name)",0, inplace=True)
    Data_Indice.replace("NA",0, inplace=True)
    Data_Indice=Data_Indice.astype(np.float64)

In [30]:
Matriz_Indices={}
for i in Todas_Indices:
    Matriz_Indices[f'{i}']=LimpiandoIndices(Indices_Datos,Fechas_Diarias,i)

In [31]:
Datos_CIQ={}
for i in Todas_Finan:
    Datos_CIQ[f'{i}']=LimpiandoCIQ(Financials_Datos,Fechas_Tri,i)
for y in Todas_Cons_Const:
    Datos_CIQ[f'{y}']=LimpiandoCIQ(Cons_Const_Datos,Fechas_Tri,y)
for z in Todas_Min_Utl:
    Datos_CIQ[f'{z}']=LimpiandoCIQ(Min_Utl_Datos,Fechas_Tri,z)
    

C:\Users\P048513\AppData\Local\Temp\ipykernel_28008\2504309530.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Data_CIQ.replace("NA",0, inplace=True)
C:\Users\P048513\AppData\Local\Temp\ipykernel_28008\2504309530.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Data_CIQ.replace("NA",0, inplace=True)
C:\Users\P048513\AppData\Local\Temp\ipykernel_28008\2504309530.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
Datos_BBG={}
for i in Todas_Finan:
    Datos_BBG[f'{i}']=LimpiandoBBG(Financials_Datos,Fechas_Diarias,i)
for i in Todas_Cons_Const:
    Datos_BBG[f'{i}']=LimpiandoBBG(Cons_Const_Datos,Fechas_Diarias,i)
for i in Todas_Min_Utl:
    Datos_BBG[f'{i}']=LimpiandoBBG(Min_Utl_Datos,Fechas_Diarias,i)

## Creacion de Multiplos

In [ ]:
Fechas_Nuevas=(Datos_BBG["BAP US Equity"].index)
def multiplos(Activo,Datos_BBG,Datos_CIQ,Fechas_Nuevas):

    EV=pd.DataFrame(Datos_BBG[Activo]["CURR_ENTP_VAL"])
    EV=EV.reset_index()
    EV["Fecha"]=EV["Fecha"].dt.to_period('Q')
    EV=EV.set_index("Fecha")
    EV=EV.squeeze()

    Market_Cap=pd.DataFrame(Datos_BBG[Activo]["CUR_MKT_CAP"])
    Market_Cap=Market_Cap.reset_index()
    Market_Cap["Fecha"]=Market_Cap["Fecha"].dt.to_period('Q')
    Market_Cap=Market_Cap.set_index("Fecha")
    Market_Cap=Market_Cap.squeeze()

    #Trail_PE
    Net_Income=pd.DataFrame(Datos_CIQ[Activo]["IQ_NI"])
    Net_Income=Net_Income.reset_index()
    Net_Income["Fecha"]=Net_Income["Fecha"].dt.to_period('Q')
    Net_Income=Net_Income.set_index("Fecha")
    Net_Income=Net_Income.squeeze()


    Trail_PE=pd.DataFrame(Market_Cap/Net_Income, columns=["Trail_PE"])
    Trail_PE["Fecha"]=Fechas_Nuevas
    Trail_PE.set_index("Fecha", inplace=True)
    Trail_PE.replace([np.inf, -np.inf], 0, inplace=True)
    #Forwards_PE
    Fwd_Net_Income=pd.DataFrame(Datos_CIQ[Activo]["Fwd_IQ_NI"])
    Fwd_Net_Income=Fwd_Net_Income.reset_index()
    Fwd_Net_Income["Fecha"]=Fwd_Net_Income["Fecha"].dt.to_period('Q')
    Fwd_Net_Income=Fwd_Net_Income.set_index("Fecha")
    Fwd_Net_Income=Fwd_Net_Income.squeeze()


    Fwd_PE=pd.DataFrame(Market_Cap/Fwd_Net_Income, columns=["Fwd_PE"])
    Fwd_PE["Fecha"]=Fechas_Nuevas
    Fwd_PE.set_index("Fecha", inplace=True)
    Fwd_PE.replace([np.inf, -np.inf], 0, inplace=True)
    #Trail_PS

    Sales=pd.DataFrame(Datos_CIQ[Activo]["IQ_TOTAL_REV"])
    Sales=Sales.reset_index()
    Sales["Fecha"]=Sales["Fecha"].dt.to_period('Q')
    Sales=Sales.set_index("Fecha")
    Sales=Sales.squeeze()


    Trail_PS=pd.DataFrame(Market_Cap/Sales, columns=["Trail_PS"])
    Trail_PS["Fecha"]=Fechas_Nuevas
    Trail_PS.set_index("Fecha", inplace=True)
    Trail_PS.replace([np.inf, -np.inf], 0, inplace=True)
    #Forwards_PS
    Fwd_Sales=pd.DataFrame(Datos_CIQ[Activo]["Fwd_IQ_TOTAL_REV"])
    Fwd_Sales=Fwd_Sales.reset_index()
    Fwd_Sales["Fecha"]=Fwd_Sales["Fecha"].dt.to_period('Q')
    Fwd_Sales=Fwd_Sales.set_index("Fecha")
    Fwd_Sales=Fwd_Sales.squeeze()


    Fwd_PS=pd.DataFrame(Market_Cap/Fwd_Sales, columns=["Fwd_PS"])
    Fwd_PS["Fecha"]=Fechas_Nuevas
    Fwd_PS.set_index("Fecha", inplace=True)
    Fwd_PS.replace([np.inf, -np.inf], 0, inplace=True)
    #Trail_PB

    Book=pd.DataFrame(Datos_CIQ[Activo]["IQ_TOTAL_EQUITY"])
    Book=Book.reset_index()
    Book["Fecha"]=Book["Fecha"].dt.to_period('Q')
    Book=Book.set_index("Fecha")
    Book=Book.squeeze()


    Trail_PB=pd.DataFrame(Market_Cap/Book, columns=["Trail_PB"])
    Trail_PB["Fecha"]=Fechas_Nuevas
    Trail_PB.set_index("Fecha", inplace=True)
    Trail_PB.replace([np.inf, -np.inf], 0, inplace=True)
    #Forwards_PB
    Fwd_Book=pd.DataFrame(Datos_CIQ[Activo]["Fwd_IQ_TOTAL_EQUITY"])
    Fwd_Book=Fwd_Book.reset_index()
    Fwd_Book["Fecha"]=Fwd_Book["Fecha"].dt.to_period('Q')
    Fwd_Book=Fwd_Book.set_index("Fecha")
    Fwd_Book=Fwd_Book.squeeze()


    Fwd_PB=pd.DataFrame(Market_Cap/Fwd_Book, columns=["Fwd_PB"])
    Fwd_PB["Fecha"]=Fechas_Nuevas
    Fwd_PB.set_index("Fecha", inplace=True)
    Fwd_PB.replace([np.inf, -np.inf], 0, inplace=True)
    #Trail_EVEBITDA

    EBITDA=pd.DataFrame(Datos_CIQ[Activo]["IQ_EBITDA"])
    EBITDA=EBITDA.reset_index()
    EBITDA["Fecha"]=EBITDA["Fecha"].dt.to_period('Q')
    EBITDA=EBITDA.set_index("Fecha")
    EBITDA=EBITDA.squeeze()


    Trail_EVEBITDA=pd.DataFrame(EV/EBITDA, columns=["Trail_EVEBITDA"])
    Trail_EVEBITDA["Fecha"]=Fechas_Nuevas
    Trail_EVEBITDA.set_index("Fecha", inplace=True)
    Trail_EVEBITDA.replace([np.inf, -np.inf], 0, inplace=True)
    #Forwards_EVEBITDA
    Fwd_EVEBITDA=Trail_EVEBITDA.copy()
    Fwd_EVEBITDA.rename(columns={"Trail_EVEBITDA":"Fwd_EVEBITDA"},inplace=True)
    
    Fwd_EBITDA=pd.DataFrame(Datos_CIQ[Activo]["Fwd_IQ_EBITDA"])
    Fwd_EBITDA=Fwd_EBITDA.reset_index()
    Fwd_EBITDA["Fecha"]=Fwd_EBITDA["Fecha"].dt.to_period('Q')
    Fwd_EBITDA=Fwd_EBITDA.set_index("Fecha")
    Fwd_EBITDA=Fwd_EBITDA.squeeze()


    Fwd_EVEBITDA=pd.DataFrame(Market_Cap/Fwd_EBITDA, columns=["Fwd_EVEBITDA"])
    Fwd_EVEBITDA["Fecha"]=Fechas_Nuevas
    Fwd_EVEBITDA.set_index("Fecha", inplace=True)
    Fwd_EVEBITDA.replace([np.inf, -np.inf], 0, inplace=True)
    
    
    df_multiplos= pd.concat([Trail_PE,Fwd_PE,Trail_PS,Fwd_PS,Trail_PB,Fwd_PB,Trail_EVEBITDA,Fwd_EVEBITDA],axis=1)
    
    return df_multiplos


In [ ]:
Matriz_Multiplos={}
for i in Todas_Finan:
    Matriz_Multiplos[f'{i}']=multiplos(i,Datos_BBG,Datos_CIQ,Fechas_Nuevas)
for i in Todas_Cons_Const:
    Matriz_Multiplos[f'{i}']=multiplos(i,Datos_BBG,Datos_CIQ,Fechas_Nuevas)
for i in Todas_Min_Utl:
    Matriz_Multiplos[f'{i}']=multiplos(i,Datos_BBG,Datos_CIQ,Fechas_Nuevas)

In [ ]:
Matriz_Multiplos_Indices={}
for i in Todas_Indices:
    Matriz_Multiplos_Indices[f'{i}']=Matriz_Indices[i].iloc[::,1:]

### Data Historica Múltiplos (Beta version)

In [ ]:
import pandas as pd
import numpy as np

multiplos_concat = pd.concat(Matriz_Multiplos, names=["Activo", "Fecha"]).reset_index()
metric_cols = [
    c for c in multiplos_concat.columns
    if c not in ["Activo", "Fecha"] and not str(c).lower().startswith("fwd_")
]

multiplos_long = multiplos_concat.melt(
    id_vars=["Activo", "Fecha"],
    value_vars=metric_cols,
    var_name="Multiplo",
    value_name="Valor"
)

multiplos_long["Fecha"] = pd.to_datetime(multiplos_long["Fecha"], errors="coerce")
multiplos_long["Valor"] = pd.to_numeric(multiplos_long["Valor"], errors="coerce")
multiplos_long.loc[multiplos_long["Valor"] == 0, "Valor"] = np.nan
multiplos_long = multiplos_long.dropna(subset=["Valor"]).reset_index(drop=True)

multiplos_wide = (
    multiplos_long
        .pivot(index="Fecha", columns=["Activo", "Multiplo"], values="Valor")
        .sort_index(axis=0)
        .sort_index(axis=1)
)
multiplos_wide.index.name = "Fecha"


diccionario_path = r"Z:\Mesa de Inversiones\Bottom-Up\5 Estrategia del Portafolio\Monitores\ZODA\ZODA.xlsm"
hoja_diccionario = "GICS"  
output_path = r"Z:\Mesa de Inversiones\Bottom-Up\5 Estrategia del Portafolio\Monitores\ZODA\Multiplos_hist.xlsx"

dic = pd.read_excel(diccionario_path, sheet_name=hoja_diccionario, dtype=str, engine="openpyxl").apply(
    lambda c: c.str.strip() if c.dtype == "object" else c
)

# Mapas: BBG -> Empresa, BBG -> GICS
map_emp = dic.set_index("BBG")["Empresa"].to_dict()
map_gics = dic.set_index("BBG")["GICS"].to_dict()

# Reemplazar BBG (Activo) por Empresa en las columnas; obtener GICS por columna
nuevas_cols = [(map_emp.get(a, a), m) for a, m in multiplos_wide.columns]
sectores = [map_gics.get(a, "Sin GICS") for a, _ in multiplos_wide.columns]
multiplos_wide.columns = pd.MultiIndex.from_tuples(nuevas_cols, names=["Empresa", "Multiplo"])

df_sec = pd.DataFrame({"Empresa": [c[0] for c in multiplos_wide.columns], "GICS": sectores}).drop_duplicates()

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for g in sorted(df_sec["GICS"].unique()):
        empresas = df_sec.loc[df_sec["GICS"] == g, "Empresa"]
        cols_sector = [c for c in multiplos_wide.columns if c[0] in empresas.values]
        multiplos_wide[cols_sector].to_excel(writer, sheet_name=str(g)[:31])

## Creacion de MarketCap

In [ ]:
MarketCap=pd.DataFrame(Datos_BBG[Todas_Finan[0]]['CUR_MKT_CAP'])
MarketCap.rename(columns={"CUR_MKT_CAP":Todas_Finan[0]},inplace=True)
for i in Todas_Finan[1:]:
    temp=pd.DataFrame(Datos_BBG[i]['CUR_MKT_CAP'])
    temp.rename(columns={"CUR_MKT_CAP":i},inplace=True)
    MarketCap= pd.concat([MarketCap,temp],axis=1)
for i in Todas_Cons_Const[1:]:
    temp=pd.DataFrame(Datos_BBG[i]['CUR_MKT_CAP'])
    temp.rename(columns={"CUR_MKT_CAP":i},inplace=True)
    MarketCap= pd.concat([MarketCap,temp],axis=1)
for i in Todas_Min_Utl:
    temp=pd.DataFrame(Datos_BBG[i]['CUR_MKT_CAP'])
    temp.rename(columns={"CUR_MKT_CAP":i},inplace=True)
    MarketCap= pd.concat([MarketCap,temp],axis=1)


# Z

In [ ]:
def Z(Prueba_Z, Anios):
    Prueba_Z.index = pd.to_datetime(Prueba_Z.index)
    window_size = Anios * 260  # número aproximado de días hábiles

    
    rolling_mean = Prueba_Z.rolling(window=window_size, min_periods=window_size).mean()
    rolling_std = Prueba_Z.rolling(window=window_size, min_periods=window_size).std()

    Z_res = (Prueba_Z - rolling_mean) / rolling_std

    return Z_res

In [ ]:
def Creacion_Z_Final (Prueba_Z):
    Matriz_de_Zs={}
    for i in Anios_Z:
        Matriz_de_Zs[f'Z_anios_{i}']= Z(Prueba_Z,i)
        Peso=Pesos_Z[Anios_Z.index(i)]
        Matriz_de_Zs[f'Z_anios_{i}']=Matriz_de_Zs[f'Z_anios_{i}']*Peso


    #Sumando Matrices
    Z_Acumulados = pd.DataFrame()
    for i in Anios_Z:
        if Z_Acumulados.empty:
            Z_Acumulados = Matriz_de_Zs[f'Z_anios_{i}']
        else:
            Z_Acumulados = Z_Acumulados.add(Matriz_de_Zs[f'Z_anios_{i}'], fill_value=0)

    ## Viendo desde cuando agarrar
    Mayor_rango= None
    for index,peso in enumerate(reversed(Pesos_Z)):
        if peso !=0:
            Ubi_Mayor_rango= len(Pesos_Z)-1-index
            break

#     print("Periodo de Z rolling más largo: "+str(Rango_Zs[Ubi_Mayor_rango])+" Años")

    Desde_aqui=Matriz_de_Zs[f'Z_anios_{Anios_Z[Ubi_Mayor_rango]}'].dropna(axis=0, how="all").index[0]
#     print("Serie disponible desde:"+ str(Desde_aqui))
    Z_Acumulados=Z_Acumulados[Z_Acumulados.index.to_list().index(Desde_aqui):]

    if es_finan:
        Pesos_mult=Pesos_mult_Finan
    else:
        Pesos_mult=Pesos_mult_Ex_Finan
        
    Z_Final=(Z_Acumulados*Pesos_mult).sum(axis=1)
    return Z_Final

In [ ]:
#########################################################################################################

In [ ]:
Matriz_Z_Financieras={}
es_finan=True
for i in Empresas_Financieras:
    Matriz_Z_Financieras[f'Z_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos[i]), columns= [f'Z_{i}'])
Matriz_Z_Cons_Const={}
es_finan=False
for i in Empresas_Cons_Const:
    Matriz_Z_Cons_Const[f'Z_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos[i]), columns= [f'Z_{i}'])
Matriz_Z_Min_Utl={}
es_finan=False
for i in Empresas_Min_Utl:
    Matriz_Z_Min_Utl[f'Z_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos[i]), columns= [f'Z_{i}'])

Matriz_Z_Indices={}
es_finan=False 
for i in Todas_Indices:
    Matriz_Z_Indices[f'{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos_Indices[i]),columns=[f'{i}'])

## Creacion de Baskets

In [ ]:
def Creacion_DF_Basket(Basket,Matriz_Multiplos, MarketCap, Fechas):
    Dataframe_Basket=pd.DataFrame(index=Fechas_Diarias)
    
    Dataframe_T_PE=pd.DataFrame(index=Fechas_Diarias)
    Dataframe_F_PE=pd.DataFrame(index=Fechas_Diarias)
    Dataframe_T_PS=pd.DataFrame(index=Fechas_Diarias)
    Dataframe_F_PS=pd.DataFrame(index=Fechas_Diarias)
    Dataframe_T_PB=pd.DataFrame(index=Fechas_Diarias)
    Dataframe_F_PB=pd.DataFrame(index=Fechas_Diarias)
    Dataframe_T_EVEBITDA=pd.DataFrame(index=Fechas_Diarias)
    Dataframe_F_EVEBITDA=pd.DataFrame(index=Fechas_Diarias)

    ##CARGANDO INFO DEL BASKETS
    
    for i in Basket:
        temp=pd.DataFrame(Matriz_Multiplos[i]["Trail_PE"])
        temp.rename(columns={"Trail_PE":i},inplace=True)
        Dataframe_T_PE=pd.concat([Dataframe_T_PE,temp],axis=1)

        temp=pd.DataFrame(Matriz_Multiplos[i]["Fwd_PE"])
        temp.rename(columns={"Fwd_PE":i},inplace=True)
        Dataframe_F_PE=pd.concat([Dataframe_F_PE,temp],axis=1)

        temp=pd.DataFrame(Matriz_Multiplos[i]["Trail_PS"])
        temp.rename(columns={"Trail_PS":i},inplace=True)
        Dataframe_T_PS=pd.concat([Dataframe_T_PS,temp],axis=1)

        temp=pd.DataFrame(Matriz_Multiplos[i]["Fwd_PS"])
        temp.rename(columns={"Fwd_PS":i},inplace=True)
        Dataframe_F_PS=pd.concat([Dataframe_F_PS,temp],axis=1)

        temp=pd.DataFrame(Matriz_Multiplos[i]["Trail_PB"])
        temp.rename(columns={"Trail_PB":i},inplace=True)
        Dataframe_T_PB=pd.concat([Dataframe_T_PB,temp],axis=1)

        temp=pd.DataFrame(Matriz_Multiplos[i]["Fwd_PB"])
        temp.rename(columns={"Fwd_PB":i},inplace=True)
        Dataframe_F_PB=pd.concat([Dataframe_F_PB,temp],axis=1)

        temp=pd.DataFrame(Matriz_Multiplos[i]["Trail_EVEBITDA"])
        temp.rename(columns={"Trail_EVEBITDA":i},inplace=True)
        Dataframe_T_EVEBITDA=pd.concat([Dataframe_T_EVEBITDA,temp],axis=1)

        temp=pd.DataFrame(Matriz_Multiplos[i]["Fwd_EVEBITDA"])
        temp.rename(columns={"Fwd_EVEBITDA":i},inplace=True)
        Dataframe_F_EVEBITDA=pd.concat([Dataframe_F_EVEBITDA,temp],axis=1)
        #################################################
    
        MarketCap_Basket=MarketCap[Basket]
        Pesos_Basket=MarketCap_Basket.div(MarketCap_Basket.sum(axis=1),axis=0)    
    
        Trail_PE_Basket_P=Pesos_Basket.multiply(Dataframe_T_PE)
        Fwd_PE_Basket_P=Pesos_Basket.multiply(Dataframe_F_PE)
        Trail_PS_Basket_P=Pesos_Basket.multiply(Dataframe_T_PS)
        Fwd_PS_Basket_P=Pesos_Basket.multiply(Dataframe_F_PS)
        Trail_PB_Basket_P=Pesos_Basket.multiply(Dataframe_T_PB)
        Fwd_PB_Basket_P=Pesos_Basket.multiply(Dataframe_F_PB)
        Trail_EVEBITDA_Basket_P=Pesos_Basket.multiply(Dataframe_T_EVEBITDA)
        Fwd_EVEBITDA_Basket_P=Pesos_Basket.multiply(Dataframe_F_EVEBITDA)
        
        Dataframe_Basket["Trail_PE"]= Trail_PE_Basket_P.sum(axis=1)
        Dataframe_Basket["Fwd_PE"]= Trail_PE_Basket_P.sum(axis=1)
        
        Dataframe_Basket["Trail_PS"]= Trail_PS_Basket_P.sum(axis=1)
        Dataframe_Basket["Fwd_PS"]= Fwd_PS_Basket_P.sum(axis=1)
        
        Dataframe_Basket["Trail_PB"]= Trail_PB_Basket_P.sum(axis=1)
        Dataframe_Basket["Fwd_PB"]= Fwd_PB_Basket_P.sum(axis=1)
        
        Dataframe_Basket["Trail_EVEBITDA"]= Trail_EVEBITDA_Basket_P.sum(axis=1)
        Dataframe_Basket["Fwd_EVEBITDA"]= Fwd_EVEBITDA_Basket_P.sum(axis=1)    
    
    return Dataframe_Basket

In [ ]:
Matriz_Baskets_Finan={}
for i in Empresas_Financieras:
    Basket=Basket_Financieras[i]
    Matriz_Baskets_Finan[f'{i}']=Creacion_DF_Basket(Basket,Matriz_Multiplos, MarketCap, Fechas_Diarias )
    
Matriz_Baskets_Cons_Const={}
for i in Empresas_Cons_Const:
    Basket=Basket_Cons_Const[i]
    Matriz_Baskets_Cons_Const[f'{i}']=Creacion_DF_Basket(Basket,Matriz_Multiplos, MarketCap, Fechas_Diarias )
    
Matriz_Baskets_Min_Utl={}
for i in Empresas_Min_Utl:
    Basket=Basket_Min_Utl[i]
    Matriz_Baskets_Min_Utl[f'{i}']=Creacion_DF_Basket(Basket,Matriz_Multiplos, MarketCap, Fechas_Diarias )

## Z basket

In [ ]:
Matriz_Z_Financieras_Basket={}
for i in Empresas_Financieras:
    Matriz_Z_Financieras_Basket[f'Z_Basket_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos[i].div(Matriz_Baskets_Finan[i])), columns= [f'Z_Basket_{i}'])

Matriz_Z_Cons_Const_Basket={}
for i in Empresas_Cons_Const:
    Matriz_Z_Cons_Const_Basket[f'Z_Basket_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos[i].div(Matriz_Baskets_Cons_Const[i])), columns= [f'Z_Basket_{i}'])

Matriz_Z_Min_Utl_Basket={}
for i in Empresas_Min_Utl:
    Matriz_Z_Min_Utl_Basket[f'Z_Basket_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos[i].div(Matriz_Baskets_Min_Utl[i])), columns= [f'Z_Basket_{i}'])    

In [ ]:
Matriz_Z_Financieras_BVL={}

for i in Empresas_Financieras:
    Matriz_Z_Financieras_BVL[f'Z_BVL_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos[i].div(Matriz_Multiplos_Indices['SPBLPGPT Index'])), columns= [f'Z_BVL_{i}'])

Matriz_Z_Cons_Const_BVL={}
for i in Empresas_Cons_Const:
    Matriz_Z_Cons_Const_BVL[f'Z_BVL_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos[i].div(Matriz_Multiplos_Indices['SPBLPGPT Index'])), columns= [f'Z_BVL_{i}'])

Matriz_Z_Min_Utl_BVL={}
for i in Empresas_Min_Utl:
    Matriz_Z_Min_Utl_BVL[f'Z_BVL_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos[i].div(Matriz_Multiplos_Indices["SPBLPGPT Index"])), columns= [f'Z_BVL_{i}'])    

Matriz_Z_Indices_MXLA={}
for i in Todas_Indices:
    if i == "MXLA Index":
        print(i)
        break
    else:
        print(i)
        Matriz_Z_Indices_MXLA[f'Z_MXLA_{i}']=pd.DataFrame(Creacion_Z_Final(Matriz_Multiplos_Indices[i].div(Matriz_Multiplos_Indices["MXLA Index"])), columns= [f'Z_MXLA_{i}'])    
    

In [ ]:
DF_Z_Financieras = pd.concat(Matriz_Z_Financieras.values(),axis=1)
DF_Z_Cons_Const = pd.concat(Matriz_Z_Cons_Const.values(),axis=1)
DF_Z_Min_Utl = pd.concat(Matriz_Z_Min_Utl.values(),axis=1)

DF_Z_Financieras_Basket = pd.concat(Matriz_Z_Financieras_Basket.values(),axis=1)
DF_Z_Cons_Const_Basket = pd.concat(Matriz_Z_Cons_Const_Basket.values(),axis=1)
DF_Z_Min_Utl_Basket = pd.concat(Matriz_Z_Min_Utl_Basket.values(),axis=1)

DF_Z_Financieras_BVL = pd.concat(Matriz_Z_Financieras_BVL.values(),axis=1)
DF_Z_Cons_Const_BVL = pd.concat(Matriz_Z_Cons_Const_BVL.values(),axis=1)
DF_Z_Min_Utl_BVL = pd.concat(Matriz_Z_Min_Utl_BVL.values(),axis=1)

DF_Z_Indices = pd.concat(Matriz_Z_Indices.values(),axis=1)
DF_Z_Indices_MXLA = pd.concat(Matriz_Z_Indices_MXLA.values(),axis=1)

## Evolución Z empresas

In [ ]:
# Fecha_Inicio_Grafs = pd.Timestamp("2017-12-31 00:00:00")

# with PdfPages(r'Z:\Mesa de Inversiones\Bottom-Up\5 Estrategia del Portafolio\Monitores\Monitor Múltiplos RVL\EvolucionZ.pdf') as pdf:
#     fig, axis = plt.subplots(4, 6, figsize=(66, 36))
#     axis = axis.flatten()
    
#     contador = 0
#     for col in DF_Z_Financieras.columns:
#         axis[contador].plot(DF_Z_Financieras[col],color='#00008B')
#         axis[contador].set_title(f'{col}')
#         axis[contador].tick_params(axis='x', rotation=90)
#         axis[contador].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
#         axis[contador].xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))      
#         contador+=1
#     for col in DF_Z_Cons_Const.columns:
#         axis[contador].plot(DF_Z_Cons_Const[col],color='#00008B')
#         axis[contador].set_title(f'{col}')
#         axis[contador].tick_params(axis='x', rotation=90)
#         axis[contador].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
#         axis[contador].xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))     
#         contador+=1
#     for col in DF_Z_Min_Utl.columns:
#         axis[contador].plot(DF_Z_Min_Utl[col],color='#00008B')
#         axis[contador].set_title(f'{col}')
#         axis[contador].tick_params(axis='x', rotation=90)
#         axis[contador].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
#         axis[contador].xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))      
#         contador+=1    
    
#     plt.tight_layout()
#     pdf.savefig(fig)
#     plt.close(fig)
    

In [ ]:
Fecha_Inicio = pd.Timestamp("2016-1-1 00:00:00")

In [ ]:
def promedio_un_anio(data):
    data_un_anio=data.last("1Y")
    return data_un_anio.mean()

In [ ]:
def std_un_anio(data):
    data_un_anio=data.last("1Y")
    return data_un_anio.std()

In [ ]:
def graficos_juntos(df_z,df_mult,empresa,fecha):
    minima_fecha=df_mult[empresa][fecha:].replace(0, np.nan).dropna(axis=0, how="all").index.min()
    maxima_fecha=df_mult[empresa].index.max()
    fig = plt.figure(figsize=(15, 20))
    fig.suptitle(f'{empresa}', fontsize=16, x= 0.01, y=0.98, ha= "left")
    # Creando el graf para el Z
    ax1 = fig.add_subplot(5, 2, (1, 2))
    ax1.plot(df_z[f'Z_{empresa}'][minima_fecha:].fillna(0), color ='#3e63ad', linewidth=2)
    ax1.tick_params(axis='x', rotation=90)
    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax1.set_xlim([minima_fecha,maxima_fecha])
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))    
    ax1.set_title('Evolución Z')

    # Añadiendo los gráficos pequeños
    data=df_mult[empresa]["Trail_PE"][minima_fecha:].fillna(0)
    ax2 = fig.add_subplot(5, 2, 3)
    ax2.plot(data,color ="#3e63ad", linewidth=2)
    ax2.axhline(y=data.mean(),color ="#00cdcd", linestyle="--", linewidth=2)
    ax2.axhline(y=promedio_un_anio(data),color ="#cfcfcf", linestyle="--", linewidth=2)
    ax2.tick_params(axis='x', rotation=90)
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax2.set_xlim([minima_fecha,maxima_fecha])
    if truncado == 1:
        if data.std() >20:
            ax2.set_ylim([0,35])        
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')) 
    ax2.set_title('Trail P/E')
    
    data=df_mult[empresa]["Fwd_PE"][minima_fecha:].fillna(0)
    ax3 = fig.add_subplot(5, 2, 4)
    ax3.plot(data,color ="#3e63ad", linewidth=2)
    ax3.axhline(y=data.mean(),color ="#00cdcd", linestyle="--", linewidth=2)
    ax3.axhline(y=promedio_un_anio(data),color ="#cfcfcf", linestyle="--", linewidth=2)
    ax3.tick_params(axis='x', rotation=90)
    ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax3.set_xlim([minima_fecha,maxima_fecha])
    if truncado == 1:
        if data.std() >20:
            ax3.set_ylim([0,35])    
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')) 
    ax3.set_title('Forward P/E')

    data=df_mult[empresa]["Trail_PS"][minima_fecha:].fillna(0)
    ax4 = fig.add_subplot(5, 2, 5)
    ax4.plot(data,color ="#3e63ad", linewidth=2)
    ax4.axhline(y=data.mean(),color ="#00cdcd", linestyle="--", linewidth=2)
    ax4.axhline(y=promedio_un_anio(data),color ="#cfcfcf", linestyle="--", linewidth=2)
    ax4.tick_params(axis='x', rotation=90)
    ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax4.set_xlim([minima_fecha,maxima_fecha])
    if truncado == 1:
        if data.std() >20:
            ax4.set_ylim([-5,20])    
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')) 
    ax4.set_title('Trail P/S')

    data=df_mult[empresa]["Fwd_PS"][minima_fecha:].fillna(0)
    ax5 = fig.add_subplot(5, 2, 6)
    ax5.plot(data,color ="#3e63ad", linewidth=2)
    ax5.axhline(y=data.mean(),color ="#00cdcd", linestyle="--", linewidth=2)
    ax5.axhline(y=promedio_un_anio(data),color ="#cfcfcf", linestyle="--", linewidth=2)
    ax5.tick_params(axis='x', rotation=90)
    ax5.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax5.set_xlim([minima_fecha,maxima_fecha])
    if truncado == 1:
        if data.std() >20:
            ax5.set_ylim([-5,20])    
    ax5.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')) 
    ax5.set_title('Forward P/S')

    data=df_mult[empresa]["Trail_PB"][minima_fecha:].fillna(0)
    ax6 = fig.add_subplot(5, 2, 7)
    ax6.plot(data,color ="#3e63ad", linewidth=2)
    ax6.axhline(y=data.mean(),color ="#00cdcd", linestyle="--", linewidth=2)
    ax6.axhline(y=promedio_un_anio(data),color ="#cfcfcf", linestyle="--", linewidth=2)
    ax6.tick_params(axis='x', rotation=90)
    ax6.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax6.set_xlim([minima_fecha,maxima_fecha])
    if truncado == 1:
        if data.std() >20:
            ax6.set_ylim([-5,20])    
    ax6.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')) 
    ax6.set_title('Trail P/B')

    data=df_mult[empresa]["Fwd_PB"][minima_fecha:].fillna(0)
    ax7 = fig.add_subplot(5, 2, 8)
    ax7.plot(data,color ="#3e63ad", linewidth=2)
    ax7.axhline(y=data.mean(),color ="#00cdcd", linestyle="--", linewidth=2)
    ax7.axhline(y=promedio_un_anio(data),color ="#cfcfcf", linestyle="--", linewidth=2)
    ax7.tick_params(axis='x', rotation=90)
    ax7.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax7.set_xlim([minima_fecha,maxima_fecha])
    if truncado == 1:
        if data.std() >20:
            ax7.set_ylim([-5,20])    
    ax7.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')) 
    ax7.set_title('Forward P/B')

    data=df_mult[empresa]["Trail_EVEBITDA"][minima_fecha:].fillna(0)
    ax8 = fig.add_subplot(5, 2, 9)
    ax8.plot(data,color ="#3e63ad", linewidth=2)
    ax8.axhline(y=data.mean(),color ="#00cdcd", linestyle="--", linewidth=2)
    ax8.axhline(y=promedio_un_anio(data),color ="#cfcfcf", linestyle="--", linewidth=2)
    ax8.tick_params(axis='x', rotation=90)
    ax8.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax8.set_xlim([minima_fecha,maxima_fecha])
    if truncado == 1:
        if data.std() >20:
            ax8.set_ylim([-5,20])    
    ax8.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')) 
    ax8.set_title('Trail EV/EBITDA')

    data=df_mult[empresa]["Fwd_EVEBITDA"][minima_fecha:].fillna(0)
    ax9 = fig.add_subplot(5, 2, 10)
    ax9.plot(data,color ="#3e63ad", linewidth=2)
    ax9.axhline(y=data.mean(),color ="#00cdcd", linestyle="--", linewidth=2, label ="Promedio Histórico")
    ax9.axhline(y=promedio_un_anio(data),color ="#cfcfcf", linestyle="--", linewidth=2, label="Promedio Último Año")
    ax9.tick_params(axis='x', rotation=90)
    ax9.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax9.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y')) 
    ax9.set_xlim([minima_fecha,maxima_fecha])
    if truncado == 1:
        if data.std() >20:
            ax9.set_ylim([-5,20])    
    ax9.set_title('Forward EV/EBITDA')

    # Ajustando
    handles, labels = [],[]
    for handle, label in (ax9.get_legend_handles_labels()):
        if label not in labels:
            handles.append(handle)
            labels.append(label)
            
    limpio_dic=dict(zip(labels,handles))
    
    fig.legend( loc='upper right')
    
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    plt.show()
    
    return fig

In [ ]:
## Si se quiere truncar los gráficos, activar la variable truncado. Si no, dejarla comentada
truncado=1
# truncado= None

In [ ]:
with PdfPages(rf'Z:\Mesa de Inversiones\Bottom-Up\5 Estrategia del Portafolio\Monitores\ZODA\PDFs\Evolucion Z y Multiplos {fecha_hoy}.pdf') as pdf:
    for i in Matriz_Multiplos.keys():
        if i in Empresas_Financieras:
            fig=graficos_juntos(DF_Z_Financieras,Matriz_Multiplos,i,Fecha_Inicio)
            pdf.savefig(fig)
            plt.close(fig)
        elif i in Empresas_Min_Utl:
            fig=graficos_juntos(DF_Z_Min_Utl,Matriz_Multiplos,i,Fecha_Inicio)
            pdf.savefig(fig)
            plt.close(fig)
        elif i in Empresas_Cons_Const:
            fig=graficos_juntos(DF_Z_Cons_Const,Matriz_Multiplos,i,Fecha_Inicio)
            pdf.savefig(fig)
            plt.close(fig)

## Multiplos

In [ ]:
# Fecha_Inicio_mult = pd.Timestamp("2020-1-1 00:00:00")

In [ ]:
# def creadora_graf_mult(nombre):
#     fig, axis = plt.subplots(2, 4, figsize=(24, 8.27))

#     axis = axis.flatten()
#     contador=0
#     contador_fwd=4
#     for col in Matriz_Multiplos[nombre].columns:
#         if contador in [0,1,2,3]:
#             axis[contador].plot(Matriz_Multiplos[nombre][col][Fecha_Inicio_mult:].fillna(0),color='#00008B')
#             axis[contador].set_title(f'{col}')
#             axis[contador].set_xlim([Matriz_Multiplos[nombre][Fecha_Inicio_mult:].index.min(),Matriz_Multiplos[nombre].index.max()])
#             axis[contador].tick_params(axis='x', rotation=90)
#             axis[contador].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
#             axis[contador].xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))

            
            
            
            
#             contador+=4
#         else:
#             axis[contador].plot(Matriz_Multiplos[nombre][col][Fecha_Inicio_mult:].fillna(0),color='#00008B')
#             axis[contador].set_title(f'{col}')
#             axis[contador].set_xlim([Matriz_Multiplos[nombre][Fecha_Inicio_mult:].index.min(),Matriz_Multiplos[nombre].index.max()])
#             axis[contador].tick_params(axis='x', rotation=90)
#             axis[contador].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
#             axis[contador].xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))

#             contador-=3
#     fig.suptitle(nombre, fontsize=32)
#     plt.subplots_adjust(top=0.85) 
#     plt.tight_layout(rect=[0, 0, 1, 0.95])
#     plt.tight_layout()
#     return fig

In [ ]:
# with PdfPages(r'Z:\Mesa de Inversiones\Bottom-Up\5 Estrategia del Portafolio\Monitores\Monitor Múltiplos RVL\EvolucionMultiplos.pdf') as pdf:
#     for i in Matriz_Multiplos.keys():
#         if i in Empresas_Financieras:
#             fig=creadora_graf_mult(i)
#             pdf.savefig(fig)
#             plt.close(fig)
#         elif i in Empresas_Min_Utl:
#             fig=creadora_graf_mult(i)
#             pdf.savefig(fig)
#             plt.close(fig)
#         elif i in Empresas_Cons_Const:
#             fig=creadora_graf_mult(i)
#             pdf.savefig(fig)
#             plt.close(fig)

In [ ]:
combinado=pd.DataFrame()

In [ ]:
def appendar(df,combinado):
    df.columns = [f'{i}_{col}' for col in df.columns]
    combinado= pd.concat([combinado,df],axis=1)
    return combinado

In [ ]:
def appendar_basket(df,combinado):
    df.columns = [f'{i}_basket_{col}' for col in df.columns]
    combinado= pd.concat([combinado,df],axis=1)
    return combinado

In [ ]:
# for i in Matriz_Multiplos.keys():
#     if i in Empresas_Financieras:
#         combinado=appendar(Matriz_Multiplos[i].fillna(0),combinado)
        
#     elif i in Empresas_Min_Utl:
#         combinado=appendar(Matriz_Multiplos[i].fillna(0),combinado)
        
#     elif i in Empresas_Cons_Const:
#         combinado=appendar(Matriz_Multiplos[i].fillna(0),combinado)
        

### Excel graficos Semestral

In [ ]:
for i in Matriz_Multiplos.keys():
    if i in Todas_Finan:
        combinado=appendar(Matriz_Multiplos[i].fillna(0),combinado)
        
    elif i in Todas_Cons_Const:
        combinado=appendar(Matriz_Multiplos[i].fillna(0),combinado)
        
    elif i in Todas_Min_Utl:
        combinado=appendar(Matriz_Multiplos[i].fillna(0),combinado)

In [ ]:
combinado_basket= pd.DataFrame()
for i in Matriz_Multiplos.keys():
    if i in Empresas_Financieras:
        combinado_basket=appendar_basket(Matriz_Baskets_Finan[i].fillna(0),combinado_basket)
        
    elif i in Empresas_Min_Utl:
        combinado_basket=appendar_basket(Matriz_Baskets_Min_Utl[i].fillna(0),combinado_basket)
        
    elif i in Empresas_Cons_Const:
        combinado_basket=appendar_basket(Matriz_Baskets_Cons_Const[i].fillna(0),combinado_basket)
        

In [ ]:
Trail_PE = [col for col in combinado.columns if 'Trail_PE' in col]
Trail_PE1 = [item for item in Trail_PE if item.split("_Trail_PE")[0] in Todas_Finan]

Trail_PB = [col for col in combinado.columns if 'Trail_PB' in col]
Trail_PB1 = [item for item in Trail_PB if item.split("_Trail_PB")[0] in Todas_Finan]

Trail_EVEBITDA = [col for col in combinado.columns if 'Trail_EVEBITDA' in col]
Trail_EVEBITDA1 = [item for item in Trail_EVEBITDA if item.split("_Trail_EVEBITDA")[0] not in Todas_Finan]

Trail_PE_basket = [col for col in combinado_basket.columns if 'Trail_PE' in col]
Trail_PE_basket1 = [item for item in Trail_PE_basket if item.split("_basket_Trail_PE")[0] in Todas_Finan]

Trail_PB_basket = [col for col in combinado_basket.columns if 'Trail_PB' in col]
Trail_PB_basket1 = [item for item in Trail_PB_basket if item.split("_basket_Trail_PB")[0] in Todas_Finan]

Trail_EVEBITDA_basket = [col for col in combinado_basket.columns if 'Trail_EVEBITDA' in col]
Trail_EVEBITDA_basket1 = [item for item in Trail_EVEBITDA_basket if item.split("_basket_Trail_EVEBITDA")[0] not in Todas_Finan]

In [ ]:
Trail_PS = [col for col in combinado.columns if 'Trail_PS' in col]
Trail_PS1 = [item for item in Trail_PS if item.split("_Trail_PS")[0] in Todas_Finan]

In [ ]:
Trail_PS_df=combinado[Trail_PS1]

In [ ]:
Trail_PS_basket = [col for col in combinado_basket.columns if 'Trail_PS' in col]
Trail_PS_basket1 = [item for item in Trail_PS_basket if item.split("_basket_Trail_PS")[0] in Todas_Finan]

In [ ]:
Trail_PS_basket_df=combinado_basket[Trail_PS_basket1]

In [ ]:
Trail_PE_df=combinado[Trail_PE1]
Trail_PB_df=combinado[Trail_PB1]
Trail_EVEBITDA_df=combinado[Trail_EVEBITDA1]


Trail_PE_basket_df=combinado_basket[Trail_PE_basket1]
Trail_PB_basket_df=combinado_basket[Trail_PB_basket1]
Trail_EVEBITDA_basket_df=combinado_basket[Trail_EVEBITDA_basket1]

In [ ]:
combinadografs= pd.concat([Trail_PE_df,Trail_PE_basket_df, Trail_PS_df,Trail_PS_basket_df,Trail_PB_df,Trail_PB_basket_df],axis=1)

In [ ]:
combinadografs= pd.concat([Trail_PE_df, Trail_EVEBITDA_df,Trail_PE_basket_df,Trail_EVEBITDA_basket_df,Trail_PB_df,Trail_PB_basket_df],axis=1)

In [ ]:
def fechas_cercanas(quarterly_dates, daily_dates):
    daily_dates = pd.DatetimeIndex(daily_dates).sort_values()
    return [daily_dates.asof(q_date) for q_date in quarterly_dates]


In [ ]:
fechas_cercanas_lista=fechas_cercanas(Fechas_Tri,combinadografs.index)

In [ ]:
combinadografs_q=combinadografs.reindex(fechas_cercanas_lista)

In [ ]:
# combinado.to_excel("BAP Basket.xlsx")

In [ ]:
ruta_mult = r'\\sppeapp00023\Inversiones\Mesa de Inversiones\Bottom-Up\5 Estrategia del Portafolio\Monitores\ZODA\Evolución de Múltiplos Comparables.xlsx'
wb1 = xw.Book(ruta_mult, update_links=False)
sheet1 =wb1.sheets["Multiplos"]
sheet1["A1"].options(pd.DataFrame, header=1).value = combinadografs_q

In [ ]:
# combinado.to_excel("Multiplos RVL Diarios Todos.xlsx",sheet_name="Multiplos")

## Exportando a Excel

In [ ]:
Base_Final_Z= pd.concat([DF_Z_Financieras,DF_Z_Cons_Const,DF_Z_Min_Utl,DF_Z_Financieras_Basket,DF_Z_Cons_Const_Basket,DF_Z_Min_Utl_Basket,DF_Z_Financieras_BVL,DF_Z_Cons_Const_BVL,DF_Z_Min_Utl_BVL,DF_Z_Indices,DF_Z_Indices_MXLA], axis=1)

In [ ]:
wb1 = xw.Book(Data, update_links=False)
sheet1 =wb1.sheets["BD"]
sheet1["A1"].options(pd.DataFrame, header=1).value = Base_Final_Z

In [ ]:
wb1 = xw.Book(Monitor, update_links=False)
sheet2 =wb1.sheets["DQ_IN"]
sheet2["A1"].options(pd.DataFrame, header=1,index=False ).value = Base_Integra

In [ ]:
FiNaL = datetime.datetime.now()

In [ ]:
        print(f'\(._.)/    ¡Corrio en {FiNaL-InIcIo}!')
        print("   |")
        print("   77")

In [ ]:
Base_Final_Z_Pivot=Base_Final_Z.stack().reset_index().copy()
# Base_Final_Z_Pivot.columns=["Fecha","Temp","Activo",,"Z"]
# Base_Final_Z_Pivot[["Var","AFP","Fondo"]]= Base_Final_Z_Pivot["Fondo"].str.split("_",expand= True)
# Base_Final_Z_Pivot=Base_Final_Z_Pivot[["Fecha","AFP","Fondo","Z"]]

In [ ]:
Base_Final_Z_Pivot["level_1"].unique()

In [ ]:
ABCDEFGHIJKLMNOPQRSTUVWXYZ

## Pelotitas

In [ ]:
# Fecha_Elegida=pd.Timestamp("2024-10-29 00:00:00")

In [ ]:
# y_dic={}
# ubicacion_fecha_elegida=Matriz_Z_Financieras_Basket['Z_Basket_BAP US Equity'][::-1].index.to_list().index(Fecha_Elegida)
# for i in Empresas_Financieras:
#     z_basket=Matriz_Z_Financieras_Basket[f'Z_Basket_{i}'][::-1].iloc[ubicacion_fecha_elegida,0]
#     y_dic[i]= z_basket

# for i in Empresas_Cons_Const:
#     z_basket=Matriz_Z_Cons_Const_Basket[f'Z_Basket_{i}'][::-1].iloc[ubicacion_fecha_elegida,0]
#     y_dic[i]= z_basket
    
# for i in Empresas_Min_Utl:
#     z_basket=Matriz_Z_Min_Utl_Basket[f'Z_Basket_{i}'][::-1].iloc[ubicacion_fecha_elegida,0]
#     y_dic[i]= z_basket
    
    
# x_dic={}

# for i in Empresas_Financieras:
#     z_basket=Matriz_Z_Financieras[f'Z_{i}'][::-1].iloc[ubicacion_fecha_elegida,0]
#     x_dic[i]= z_basket
# for i in Empresas_Cons_Const:
#     z_basket=Matriz_Z_Cons_Const[f'Z_{i}'][::-1].iloc[ubicacion_fecha_elegida,0]
#     x_dic[i]= z_basket
# for i in Empresas_Min_Utl:
#     z_basket=Matriz_Z_Min_Utl[f'Z_{i}'][::-1].iloc[ubicacion_fecha_elegida,0]
#     x_dic[i]= z_basket
    
    
# y= pd.DataFrame.from_dict(y_dic, orient="index")
# x= pd.DataFrame.from_dict(x_dic, orient="index")
   
# labels=x.index.to_list()  


In [ ]:
# x[0] = pd.to_numeric(x[0], errors='coerce')
# y[0] = pd.to_numeric(y[0], errors='coerce')

In [ ]:

# fig, ax = plt.subplots()
# ax.scatter(x[0], y[0])

# for i, label in enumerate(labels):
#     if pd.notna(x[0].iloc[i]) and pd.notna(y[0].iloc[i]):
#         ax.annotate(label, (x[0].iloc[i], y[0].iloc[i]), textcoords="offset points", xytext=(0,15), ha='center')

# ax.set(xlim=(-1.5, 1.5),
#        ylim=(-2, 2))

# plt.axhline(0, color='grey', linewidth=0.5)
# plt.axvline(0, color='grey', linewidth=0.5)

# plt.xlabel('Valuation relativo stock')
# plt.ylabel('Valuation relativo Basket')
# plt.title("Valuation Relativo Acciones BVL")
# plt.show()


#### def LTM

In [ ]:
# def backwards_LTM(Fechas_BD, Base, name):
#     Fechas_BD = pd.Series(Fechas_BD)
    
#     Actual_F = pd.DataFrame(Fechas_BD, columns=["Fecha"])
#     Actual = Base.iloc[:, 0].values
    
#     offsets = [3, 6, 9]
#     # crea una lista de tamaño de las fechas y el numero de offsets
#     results = {offset: [None] * len(Fechas_BD) for offset in offsets}
    
#     # diccionario de fechas (más rapido que lista, no itera uno por uno hasta que la encuentra)
#     date_to_index = {date: idx for idx, date in enumerate(Fechas_BD)}
    
#     for idx, dia in enumerate(Fechas_BD):
#         for offset in offsets:
#             backward_date = dia - pd.DateOffset(months=offset)
#             backward_date = backward_date.replace(hour=0, minute=0, second=0)
            
#             if backward_date in date_to_index:
#                 data = Base.iloc[date_to_index[backward_date], 0]
#             else:
#                 previous_dates = Fechas_BD[Fechas_BD < backward_date]
#                 if not previous_dates.empty:
#                     closest_date = previous_dates.max()
#                     data = Base.iloc[date_to_index[closest_date], 0]
#                 else:
#                     data = None
            
#             results[offset][idx] = data
    
    
#     Tres_meses = pd.DataFrame(results[3], columns=["Tres_meses"])
#     Seis_meses = pd.DataFrame(results[6], columns=["Seis_meses"])
#     Nueve_meses = pd.DataFrame(results[9], columns=["Nueve_meses"])
    
    
#     combinada = pd.concat([pd.DataFrame(Actual, columns=["Actual"]), Tres_meses, Seis_meses, Nueve_meses], axis=1)
    
#     Sumado = combinada.sum(axis=1).to_frame(name)
    
#     Final_Sumado = pd.concat([Actual_F, Sumado], axis=1)
#     Final_Sumado.set_index("Fecha", inplace=True)
    
#     return Final_Sumado

#### def Forwards

## Creados los Dataframes individuales

In [ ]:
# def Creacion_DF_Individual(Activo ,Trail_PE,Fwd_PE,Trail_PS,Fwd_PS,Trail_PB,Fwd_PB,Trail_EVEBITDA,Fwd_EVEBITDA, Fechas):
#         Dataframe=pd.DataFrame( index=Fechas)
        
#         Dataframe["Trail_PE"]= Trail_PE[Activo]
#         Dataframe["Fwd_PE"]= Fwd_PE[Activo]
        
#         Dataframe["Trail_PS"]= Trail_PS[Activo]
#         Dataframe["Fwd_PS"]= Fwd_PS[Activo]
        
#         Dataframe["Trail_PB"]= Trail_PB[Activo]
#         Dataframe["Fwd_PB"]= Fwd_PB[Activo]
        
#         Dataframe["Trail_EVEBITDA"]= Trail_EVEBITDA[Activo]
#         Dataframe["Fwd_EVEBITDA"]= Fwd_EVEBITDA[Activo]
        
#         return Dataframe

In [ ]:
# Matriz_Empresas_Finan={}
# for i in Empresas_Financieras:
#     Matriz_Empresas_Finan[f'{i}']=Creacion_DF_Individual(i,Trail_PE_Finan,Fwd_PE_Finan,Trail_PS_Finan, Fwd_PS_Finan,Trail_PB_Finan,Fwd_PB_Finan,Trail_EVEBITDA_Finan,Fwd_EVEBITDA_Finan, Fechas )